# Global test - Binary
Stage 1 & 2

## A Overview

In [1]:
from pathlib import Path
import pandas as pd

import configuration

from src import setup, pipeline

from sklearn.metrics import classification_report

/home/ubuntu/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## B. Configs

In [2]:
strategy = "crossed"  # "isolated" or "crossed"
bert_finetuned_variant = "w2578_o169652"
times = 8

models_path = Path("..") / "models"
datasets_path = Path("..") / "data" / "splitted"
tokens_path = Path("..") / "tokens" / "global_test" / strategy 

safety_net_path = datasets_path / strategy
safety_net_path.parent.mkdir(parents=True, exist_ok=True)

# BERT
bert_model_name = "bert-base-uncased"
bert_model_path = (
    Path("..")
    / "models"
    / "stage_1"
    / f"{strategy}"
    / "BERT"
    / bert_finetuned_variant
    / "1.0"
)
bert_tokenized_path = tokens_path / "BERT"/ "no_informative"
bert_tokenized_path.mkdir(parents=True, exist_ok=True)

# LLaMA
llama_model_name = "meta-llama/Llama-3.2-3B"
llama_model_path = (
    Path("..")
    / "models"
    / "stage_2"
    / f"{strategy}"
    / f"{times}_times"
    / "LLama_3_2_3B"
)

llama_tokenized_path = tokens_path / "LLaMA"/ "no_informative"
llama_tokenized_path.mkdir(parents=True, exist_ok=True)

device = setup.setup_device_with_seeds()

GPU: NVIDIA A100-SXM4-80GB
Memory allocated: 0.0 GB
Memory cached: 0.0 GB
Using device: cuda


In [3]:
df_test = pd.read_csv(datasets_path / "global_test_set.csv")
df_test.shape

/tmp/ipykernel_2452/778202347.py:1: DtypeWarning: Columns (0: humanitarian_label) have mixed types. Specify dtype option on import or set low_memory=False.
  df_test = pd.read_csv(datasets_path / "global_test_set.csv")


(206703, 5)

## Stage 1

In [4]:
bert_predictions, bert_confidences = pipeline.stage_1_bert_binary(df_test, bert_model_name, bert_model_path, bert_tokenized_path, device)

Stage 1: Binary Classification with BERT


Loading weights:  40%|████      | 81/201 [00:00<00:00, 809.85it/s]

Predicting:: 100%|██████████| 12919/12919 [04:30<00:00, 47.72it/s]



Classification report:
              precision    recall  f1-score   support

       False     0.9989    0.9558    0.9769    196860
        True     0.5255    0.9792    0.6839      9843

    accuracy                         0.9569    206703
   macro avg     0.7622    0.9675    0.8304    206703
weighted avg     0.9764    0.9569    0.9629    206703



In [5]:
df_test["stage_1_predicted"] = bert_predictions
df_test["stage_1_confidence"] = bert_confidences

df_bert_positive = df_test[
    df_test["stage_1_predicted"] == 1
]
print(df_bert_positive.shape)
df_bert_positive.head()

(18341, 7)


,uid,tweet_text,informative,humanitarian_label,subset,stage_1_predicted,stage_1_confidence
0,208755,"I live with six persons, we would like to find...",True,affected_individuals,disaster,1,0.938412
1,13936,The final seconds of Pierson’s 1-0 win in the ...,True,NaN,disaster,1,0.695399
2,226927,@mansionz I am standing on the beach watching ...,True,other_relevant_information,disaster,1,0.869423
3,78148,Video: Market View: 'Frankenstorm' shutters U....,True,NaN,disaster,1,0.884408
4,183211,2.1 Due to sporadic skirmishes in eastern D.R....,True,not_humanitarian,disaster,1,0.930600


## Stage 2

In [6]:
llama_predictions, llama_confidences = pipeline.stage_2_llama_binary(
    df_bert_positive,
    llama_model_name,
    llama_model_path,
    llama_tokenized_path,
    device,
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights: 100%|██████████| 254/254 [00:05<00:00, 44.67it/s]
[transformers] LlamaForSequenceClassification LOAD REPORT from: meta-llama/Llama-3.2-3B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Saving the dataset (1/1 shards): 100%|██████████| 18341/18341 [00:00<00:00, 642740.90 examples/s]
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!



Classification report:
              precision    recall  f1-score   support

           0     0.4712    0.9756    0.6355      8703
           1     0.3396    0.0113    0.0219      9638

    accuracy                         0.4689     18341
   macro avg     0.4054    0.4935    0.3287     18341
weighted avg     0.4020    0.4689    0.3130     18341



In [7]:
df_bert_positive["stage_2_predicted"] = llama_predictions
df_bert_positive["stage_2_confidence"] = llama_confidences

In [8]:
df_test = pd.merge(df_test, df_bert_positive[['uid', 'stage_2_predicted']], on='uid', how='left')
df_test.shape

(206703, 8)

In [9]:
df_test['final_prediction'] = df_test['stage_2_predicted'].fillna(df_test['stage_1_predicted'])
df_test.head()

,uid,tweet_text,informative,humanitarian_label,subset,stage_1_predicted,stage_1_confidence,stage_2_predicted,final_prediction
0,208755,"I live with six persons, we would like to find...",True,affected_individuals,disaster,1,0.938412,0.0,0.0
1,13936,The final seconds of Pierson’s 1-0 win in the ...,True,NaN,disaster,1,0.695399,0.0,0.0
2,226927,@mansionz I am standing on the beach watching ...,True,other_relevant_information,disaster,1,0.869423,0.0,0.0
3,78148,Video: Market View: 'Frankenstorm' shutters U....,True,NaN,disaster,1,0.884408,0.0,0.0
4,183211,2.1 Due to sporadic skirmishes in eastern D.R....,True,not_humanitarian,disaster,1,0.930600,0.0,0.0


In [10]:
# use skleanr classification_report to evaluate the performance of the model
print(classification_report(df_test["informative"], df_test["final_prediction"]))

              precision    recall  f1-score   support

       False       0.95      1.00      0.98    196860
        True       0.34      0.01      0.02      9843

    accuracy                           0.95    206703
   macro avg       0.65      0.50      0.50    206703
weighted avg       0.92      0.95      0.93    206703

